## ↓のデータフレーム変換関数を定義したセルを実行済みの状態で進めてください。

In [ ]:
import requests
import time
import pandas as pd

def gen_boj_dataframe(
    db:str,
    codes:list[str],
    startdate:str,
    enddate:str) -> pd.DataFrame:
    url = "https://www.stat-search.boj.or.jp/api/v1/getDataCode"
    params = {
        "DB": db,
        "CODE": ",".join(codes),
        "FORMAT": "JSON",
        "LANG":"JP",
        "STARTDATE": startdate,
        "ENDDATE": enddate
        }

    # ページネーションで最後のページを取得するまで、繰り返し処理を実行
    # Web APIからの取得データを格納
    all_results = []

    while True:
        # Web APIへのリクエストと例外処理。エラー時は空のデータフレームを返して終了
        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            response_data = response.json()
        except requests.exceptions.Timeout:
            print("Web APIから30秒以内に応答がありませんでした。")
            return pd.DataFrame()
        except requests.exceptions.ConnectionError:
            print("Web APIに接続できませんでした。")
            return pd.DataFrame()
        except requests.exceptions.HTTPError as error:
            print(f"HTTPエラーが発生しました: {error}")

            content_type = response.headers.get("Content-Type", "")
            if "application/json" in content_type:
                error_data = response.json()
                print(error_data["MESSAGE"])
            return pd.DataFrame()
        except requests.exceptions.JSONDecodeError:
            print("レスポンスをJSONとして読み込めませんでした。")
            return pd.DataFrame()
        else:
            if response_data["MESSAGEID"] == "M181030I":
                print(response_data["MESSAGE"])
                return pd.DataFrame()

            all_results.extend(response_data["RESULTSET"])
            next_position = response_data["NEXTPOSITION"]
            if next_position is None:
                break

            params["STARTPOSITION"] = next_position
            time.sleep(1)

    # 取得結果をデータフレームに変換
    df = pd.DataFrame(all_results)
    result_df = pd.DataFrame()

    for i in range(len(df)):
        values_df = pd.DataFrame(df.loc[i,"VALUES"])
        values_df = values_df.assign(
            SERIES_CODE=df.loc[i, "SERIES_CODE"],
            NAME_OF_TIME_SERIES_J=df.loc[i, "NAME_OF_TIME_SERIES_J"],
            CATEGORY_J=df.loc[i, "CATEGORY_J"]
        )
        result_df = pd.concat([result_df, values_df])

    result_df = result_df[["SERIES_CODE","NAME_OF_TIME_SERIES_J","CATEGORY_J","SURVEY_DATES","VALUES"]]

    result_df = result_df.reset_index(drop=True)

    result_df = result_df.astype({'SURVEY_DATES': str})

    return result_df

## 折れ線グラフを作ろう

In [ ]:
import plotly.express as px

# 関数部分は省略

# 2020年基準の国内企業物価指数総平均データをデータフレームに変換する
codes = ["PRCG20_2200000000"]

line_df = gen_boj_dataframe("PR01",codes,"202504","202603")

# データフレームをグラフ化する
fig = px.line(
    line_df,
    x="SURVEY_DATES",
    y="VALUES",
    color="NAME_OF_TIME_SERIES_J",
    markers=True,
    title="国内企業物価指数総平均（2020年基準）",
    labels={
        "SURVEY_DATES": "時期",
        "VALUES": "物価指数総平均",
        "NAME_OF_TIME_SERIES_J": "系列名",
    },
)
fig.show()

## 棒グラフを作ろう

In [ ]:
import plotly.express as px

# 関数部分は省略

codes= ["BPBP6JYNEX","BPBP6JYNIM"]

bar_df = gen_boj_dataframe("BP01",codes,"202604","202606")

fig = px.bar(
    bar_df,
    x="SURVEY_DATES",
    y="VALUES",
    color="NAME_OF_TIME_SERIES_J",
    title="2026年4月から6月までの貿易収支の推移",
    labels={
        "NAME_OF_TIME_SERIES_J": "系列名",
        "VALUES": "金額（億円）",
        "SURVEY_DATES": "時期"
    },
    barmode="group",
)
fig.show()

## 折れ線グラフと積み上げ棒グラフを組み合わせよう

In [ ]:
import plotly.graph_objects as go

# 関数部分は省略

# データの取得・データフレームへの変換
codes= ["BPBP6JYNEX","BPBP6JYNIM","BPBP6JYNTB"]

combined_df = gen_boj_dataframe("BP01",codes,"202404","202603")

combined_df.loc[combined_df["SERIES_CODE"] == "BPBP6JYNIM", "VALUES"] *= -1

# 複合グラフを表示しやすいように横長形式のデータフレームへ変換
wide_df = combined_df.pivot(
    index="SURVEY_DATES",
    columns="NAME_OF_TIME_SERIES_J",
    values="VALUES"
).reset_index()

fig = go.Figure()

# 内訳の積み上げ棒グラフの作成
for col_name in ["貿易収支/輸出","貿易収支/輸入"]:
  fig.add_trace(
        go.Bar(
            x=wide_df["SURVEY_DATES"],
            y=wide_df[col_name],
            name=col_name,
        ),
    )

# 貿易収支合計の折れ線グラフ作成
fig.add_trace(
    go.Scatter(
        x=wide_df["SURVEY_DATES"],
        y=wide_df["貿易収支/ネット"],
        mode="lines+markers",
        name="貿易収支/ネット",
        line={"color": "black", "width": 3},
    )
)

# 棒グラフの積み上げ方法とグラフ全体の表示を設定
fig.update_layout(
    title="貿易収支の推移と内訳",
    xaxis_title="年月",
    yaxis_title="億円",
    barmode="relative",
    hovermode="x unified",
)

fig.show()

## ヒートマップを作ろう

In [ ]:
import plotly.express as px

# データの取得・データフレームへの変換
codes= ["TK99F1000601GCQ00000","TK99F1000601GCQ10000","TK99F2000601GCQ00000","TK99F2000601GCQ10000"]

heatmap_df = gen_boj_dataframe("CO",codes,"202602","202602")

# 業種列を追加
heatmap_df["業種"] = (
    heatmap_df["SERIES_CODE"]
    .str[5:9]
    .map({
        "1000": "製造業",
        "2000": "非製造業"
    })
)

# 実績予測列を追加
heatmap_df["実績予測"] = (
    heatmap_df["SERIES_CODE"]
    .str[15]
    .map({
        "0": "実績",
        "1": "予測"
    })
)

# ヒートマップ作成のための横長形式のデータフレームへ変換
heatmap_data = heatmap_df.pivot(
    index="業種",
    columns="実績予測",
    values="VALUES",
)

# ヒートマップの作成
fig = px.imshow(
    heatmap_data,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    labels={
        "x": "実績予測",
        "y": "業種",
        "color": "業況判断D.I.",
    },
    title="業種・実績予想別の業況判断D.I.",
)
fig.show()